# DDSketch example

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

from datasketches import ddsketch_dense as ddsketch
from datasketches import tdigest_double
import seaborn as sns

In [ ]:
np.random.seed(42)

stream_size = int(10e6)
plot_size = int(1e5)

### DDSketch vs t-digest: Pareto distribution

In [ ]:
alpha = 1
pareto = (np.random.pareto(5, size=plot_size) + 1) * alpha

plt.hist(pareto, bins=200, density=True)
plt.vlines(np.mean(pareto), ymin=0, ymax=5, color='r', label="approx mean")
plt.vlines(np.quantile(pareto, 0.5), ymin=0, ymax=5, color='b', label="approx 0.5 quantile")
plt.vlines(np.pow(2, 1/5) , ymin=0, ymax=5, color='g', label="true 0.5 quantile")
sns.kdeplot(pareto, color="black", label="Pareto distribution")
plt.xlim((-0, 4))
plt.legend()

In [ ]:
pareto = (np.random.pareto(5, size=stream_size) + 1) * alpha

In [ ]:
import time 
sk = ddsketch(0.004)
t0 = time.time()
for v in pareto:
  sk.update(v)
t1 = time.time()
print((t1 - t0) / len(pareto))

In [ ]:
print(time.time())

In [ ]:
print(sk.get_quantile(0.5), np.quantile(pareto, 0.5), np.pow(2, 1/5))
print(sk.get_quantile(0.25), np.quantile(pareto, 0.25))
print(str.upper("initial sketch\n"), sk)
buf = sk.serialize()
print(len(buf), sk.get_serialized_size_bytes())
deserialized_sk = ddsketch.deserialize(buf)
print(str.upper("deserialized sketch\n"), deserialized_sk)
print(sk.get_quantile(0.5), deserialized_sk.get_quantile(0.5))

In [ ]:
td = tdigest_double()
for val in pareto:
  td.update(val)

In [ ]:
print(td.get_quantile(0.5), np.quantile(pareto, 0.5), np.pow(2, 1/5))
print(td.get_quantile(0.25), np.quantile(pareto, 0.25))
print(str.upper("initial sketch\n"), td)
buf = td.serialize()
print(len(buf), td.get_serialized_size_bytes())
deserialized_td = tdigest_double.deserialize(buf)
print(str.upper("deserialized sketch\n"), deserialized_td)
print(td.get_quantile(0.5), deserialized_td.get_quantile(0.5))

## DDSketch vs t-digest: Gaussian distribution

In [ ]:
gaussian = np.random.normal(0, 1, size=plot_size)


plt.hist(gaussian, bins=100, density=True)
plt.vlines(np.mean(gaussian), ymin=0, ymax=0.5, color='r', label="approx mean")
plt.vlines(np.quantile(gaussian, 0.5), ymin=0, ymax=0.5, color='b', label="approx 0.5 quantile")
sns.kdeplot(gaussian, color="black", label="Gaussian distribution")
plt.xlim((-5, 5))
plt.legend()

In [ ]:
gaussian = np.random.normal(0, 1, size=stream_size)

In [ ]:
sk = ddsketch(0.005)
for v in gaussian:
  sk.update(v)

In [ ]:
print(sk.get_quantile(0.5), np.quantile(gaussian, 0.5))
print(sk.get_quantile(0.25), np.quantile(gaussian, 0.25))
print(str.upper("initial sketch\n"), sk)
buf = sk.serialize()
print(len(buf), sk.get_serialized_size_bytes())
deserialized_sk = ddsketch.deserialize(buf)
print(str.upper("deserialized sketch\n"), deserialized_sk)
print(sk.get_quantile(0.5), deserialized_sk.get_quantile(0.5))

In [ ]:
td = tdigest_double()
for val in gaussian:
  td.update(val)

In [ ]:
print(td.get_quantile(0.5), np.quantile(gaussian, 0.5))
print(td.get_quantile(0.25), np.quantile(gaussian, 0.25))
print(str.upper("initial sketch\n"), td)
buf = td.serialize()
print(len(buf), td.get_serialized_size_bytes())
deserialized_td = tdigest_double.deserialize(buf)
print(str.upper("deserialized sketch\n"), deserialized_td)
print(td.get_quantile(0.5), deserialized_td.get_quantile(0.5))

In [ ]:
x = np.linspace(-5, 5, 100)


print(x.tolist())
pmf = sk.get_pmf(x)
print(sk.get_rank(6))
print(pmf)

fig = plt.figure()
plt.plot(x, pmf[:-1])
plt.show()

In [ ]:
x = np.linspace(-5, 5, 100)


print(x.tolist())
pmf = td.get_pmf(x)
print(td.get_rank(6))
print(pmf)

fig = plt.figure()
plt.plot(x, pmf[:-1])
plt.show()